# DIPCOMA: Group–Ranking–Search Hybrid Band Selection and Compression

**One-run Kaggle pipeline.** Attach `krrishshandilya/dipcoma-trainval` using *Add Input* before running all cells.

This notebook combines:
1. **Group-wise selection:** K-means groups spectral bands using a correlation-equivalent Euclidean embedding.
2. **Ranking:** variance and Shannon entropy score bands; within each cluster, the highest-ranked bands form a candidate pool.
3. **Search:** genetic algorithm (GA) and binary particle-swarm optimisation (BPSO) search the candidate pool using a rate-aware reconstruction objective.

The group → rank → search sequence follows the hybrid strategy described by Vaddi et al. (2024), pp. 86–87 (their Algorithm 2 is specifically clustering then ranking within clusters). The search stage and rate-aware codec term are implemented as this project's extension; they are not claimed to be the paper's exact algorithm.

**Evaluation:** compression ratio, bpp, PSNR, 2-D SSIM averaged over bands, SAM (degrees), MAE, mean relative error, retained bands, redundancy, entropy, OA/AA/κ when pixel labels genuinely exist, encode/decode time, regression parameter count, model size and peak memory. These follow the uploaded *Sentinel-2 Compression: Approaches, Metrics and Publishability* note, pp. 4–5. Metrics are computed in the original 13-band space.

**Important protocol:** your `REGIONS` table is authoritative for the intended train/val/test regions. Training regions are used for fitting/search; validation regions select the final candidate; test regions are held out for final reporting. If the dataset has no official test split, the notebook reserves one validation region for model selection and evaluates on the remaining validation region(s), explicitly labelling this as a held-out validation-region proxy rather than an official test result. It never silently relabels validation data as test data. The two month strings are retained as metadata and are not used as image paths because the uploaded notebook/data format does not establish a month-to-file mapping.

Run the notebook with **Run All**. Results are written to `/kaggle/working/hybrid_dipcoma_outputs/` and zipped.


In [ ]:
import os, re, json, math, time, glob, random, shutil, itertools, platform, sys, gc, tracemalloc, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import cohen_kappa_score, accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from skimage.metrics import structural_similarity
from PIL import Image, features
import io

SEED = 42
np.random.seed(SEED); random.seed(SEED)
OUT = Path("/kaggle/working/hybrid_dipcoma_outputs")
for d in ["tables","figures","selections","models","reports"]: (OUT/d).mkdir(parents=True, exist_ok=True)

# Exact region plan supplied for this experiment.
REGIONS = [
("train","Central Valley",36.5,-120.0,"2025-07","2025-04"),
("train","Iowa",42.0,-93.5,"2025-07","2025-10"),
("train","Punjab",30.9,75.5,"2025-02","2025-11"),
("train","Amazon Manaus",-3.1,-60.0,"2025-08","2025-06"),
("train","Central Borneo",1.0,114.0,"2025-07","2025-09"),
("train","Black Forest",48.0,8.2,"2025-07","2025-10"),
("train","Sahara Algeria",30.0,8.0,"2025-01","2025-07"),
("train","Atacama",-23.5,-68.5,"2025-01","2025-07"),
("train","Serengeti",-2.5,34.8,"2025-07","2025-02"),
("train","Swiss Alps",46.5,8.0,"2025-07","2025-09"),
("train","Iceland Vatnajokull",64.4,-17.5,"2025-07","2025-09"),
("train","Delhi",28.6,77.2,"2025-03","2025-11"),
("train","Lake Victoria",-1.0,33.0,"2025-06","2025-01"),
("train","Sundarbans",22.0,89.0,"2025-02","2025-11"),
("train","Maldives",4.0,73.4,"2025-02","2025-04"),
("val","Murray-Darling",-34.5,146.0,"2025-01","2025-07"),
("val","Tokyo Bay",35.7,139.8,"2025-11","2025-02"),
("val","Gobi",43.0,104.0,"2025-07","2025-10"),
("test","Po Valley",45.0,10.5,"2025-07","2025-04"),
("test","Northern Sweden",64.5,19.0,"2025-07","2025-09"),
("test","Rub al Khali",20.5,51.5,"2025-01","2025-07"),
("test","Everest region",28.0,86.9,"2025-10","2025-04"),
("test","Lagos",6.5,3.4,"2025-01","2025-11"),
("test","Great Barrier Reef",-18.5,147.0,"2025-07","2025-10"),
]
CFG = dict(n_bands=13, k_values=[3,4,5,6,7,8,10], main_k=6,
           pixels_per_patch_fit=2500, max_fit_patches_per_region=8,
           max_search_patches_per_region=4, max_eval_patches_per_region=12,
           sample_pixels=10000, cluster_candidates_per_group=3,
           ga_population=12, ga_generations=8, pso_particles=12, pso_iterations=8,
           ga_pso_seeds=2, codec_rate=8, mu=0.5, nu=0.1, gamma=0.05,
           max_test_patches_per_region=20, ssim_max_side=256,
           original_bits=16, psnr_max=65535.0)
print("Python",platform.python_version(),"| NumPy",np.__version__,"| JPEG2000:",features.check("jpg_2000"))
assert features.check("jpg_2000"), "Pillow JPEG2000/OpenJPEG support is required by the compression evaluation."
print("Output:",OUT)


In [ ]:
# ---------------------------- locate and inspect dataset ----------------------------
def norm_name(s): return re.sub(r"[^a-z0-9]+", "", s.lower())

def find_root():
    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("/kaggle/input does not exist. Attach the DIPCOMA dataset using Kaggle Add Input.")
    candidates = []
    for p in [base] + [x for x in base.rglob("*") if x.is_dir()]:
        if (p / "train").is_dir() and (p / "val").is_dir():
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Could not find a dataset directory containing both train/ and val/. Attach dipcoma-trainval via Kaggle Add Input.")
    # Prefer the deepest matching root (the actual dataset directory, not a parent mount).
    return max(candidates, key=lambda p: len(p.parts))

DATA_ROOT = find_root()
print("Dataset root:", DATA_ROOT)
for split in ["train", "val", "test"]:
    p = DATA_ROOT / split
    n = len([x for x in p.iterdir() if x.is_dir()]) if p.is_dir() else 0
    print(f"{split} exists: {p.is_dir()} | region dirs: {n}")

REGION_META = {norm_name(r[1]): r for r in REGIONS}
def discover_region_dirs(split):
    base = DATA_ROOT / split
    if not base.is_dir(): return {}
    return {norm_name(d.name): d for d in base.iterdir() if d.is_dir()}

def get_region_files(split, region_name):
    dirs = discover_region_dirs(split)
    key = norm_name(region_name)
    d = dirs.get(key)
    if d is None:
        hits = [v for k, v in dirs.items() if key in k or k in key]
        if len(hits) == 1: d = hits[0]
    if d is None: return []
    return sorted(p for p in d.rglob("*.npy") if p.is_file())

region_rows = []
region_files = {}
for split, name, lat, lon, m1, m2 in REGIONS:
    files = get_region_files(split, name)
    region_files[(split, name)] = files
    region_rows.append(dict(split=split, region=name, latitude=lat, longitude=lon,
                            metadata_month_1=m1, metadata_month_2=m2, n_patches=len(files),
                            matched=bool(files), directory=str(files[0].parent) if files else ""))
inventory_df = pd.DataFrame(region_rows)
inventory_df.to_csv(OUT / "tables" / "region_inventory.csv", index=False)
display(inventory_df)

if not any(r["n_patches"] for r in region_rows if r["split"] == "train"):
    raise RuntimeError("No supplied train regions matched dataset folders. Inspect tables/region_inventory.csv and the actual region directory names.")


def load_patch(path):
    a = np.load(path, allow_pickle=False)
    if a.ndim != 3:
        raise ValueError(f"{path}: expected a 3-D (13,H,W) or (H,W,13) array, got {a.shape}")
    if a.shape[0] == CFG["n_bands"]: pass
    elif a.shape[-1] == CFG["n_bands"]: a = np.moveaxis(a, -1, 0)
    else: raise ValueError(f"{path}: cannot identify 13-band axis in {a.shape}")
    return np.nan_to_num(a.astype(np.float32), nan=0, posinf=0, neginf=0)

# Keep the supplied train and validation regions intact. Never relabel validation as official test.
train_fit, train_search, val_paths, test_paths = [], [], [], []
train_region_count = 0
for split, name, *_ in REGIONS:
    files = region_files[(split, name)]
    if not files: continue
    if split == "train":
        train_region_count += 1
        ids = np.random.default_rng(SEED + train_region_count).permutation(len(files))
        nfit = min(CFG["max_fit_patches_per_region"], max(1, len(files) // 2))
        fit = [files[i] for i in ids[:nfit]]
        rest = [files[i] for i in ids[nfit:]]
        nsearch = min(CFG["max_search_patches_per_region"], len(rest))
        train_fit.extend(fit)
        train_search.extend(rest[:nsearch])
    elif split == "val":
        ids = np.random.default_rng(SEED + len(val_paths) + 1000).permutation(len(files))
        val_paths.extend([files[i] for i in ids[:CFG["max_eval_patches_per_region"]]])
    elif split == "test":
        ids = np.random.default_rng(SEED + len(test_paths) + 2000).permutation(len(files))
        test_paths.extend([files[i] for i in ids[:CFG["max_test_patches_per_region"]]])

print(f"Initial patches: fit={len(train_fit)}, search={len(train_search)}, validation={len(val_paths)}, official_test={len(test_paths)}")
if not train_fit or not train_search:
    raise RuntimeError("Need disjoint train-fit and train-search patches. Increase the number of patches or adjust per-region sampling limits.")
if not val_paths:
    raise RuntimeError("No validation patches matched the supplied validation regions. See region_inventory.csv; refusing to substitute training data.")

# If Kaggle provides no official test split (as in the observed run), split the THREE
# supplied validation regions by REGION: one region for model selection, remaining region(s)
# for a clearly-labelled held-out validation-region evaluation. This is NOT called an official test set.
val_region_names = [name for split, name, *_ in REGIONS if split == "val" and region_files[(split, name)]]
val_selection_paths, heldout_val_paths = [], []
if not test_paths:
    if len(val_region_names) < 2:
        raise RuntimeError("Dataset has no official test split and fewer than two matched validation regions. A leakage-safe final evaluation cannot be formed; add an independent test region.")
    selection_region = val_region_names[0]
    heldout_regions = val_region_names[1:]
    val_selection_paths = region_files[("val", selection_region)][:CFG["max_eval_patches_per_region"]]
    for name in heldout_regions:
        heldout_val_paths.extend(region_files[("val", name)][:CFG["max_test_patches_per_region"]])
    if not val_selection_paths or not heldout_val_paths:
        raise RuntimeError("Could not form non-empty region-disjoint validation-selection and held-out evaluation sets.")
    val_paths = val_selection_paths
    final_eval_paths = heldout_val_paths
    evaluation_label = "Held-out validation regions (proxy; dataset has no official test split)"
else:
    final_eval_paths = test_paths
    evaluation_label = "Official test split"

print(f"Final protocol: train_fit={len(train_fit)}, train_search={len(train_search)}, "
      f"model_selection_val={len(val_paths)}, final_evaluation={len(final_eval_paths)}")
print("Final evaluation protocol:", evaluation_label)
for p in (train_fit[:2] + val_paths[:1] + final_eval_paths[:1]):
    a = load_patch(p)
    print(p.relative_to(DATA_ROOT), a.shape, a.dtype, float(a.min()), float(a.max()))


In [ ]:
# ---------------------------- band feature engineering ----------------------------
def sample_band_matrix(paths, max_pixels=CFG["sample_pixels"], seed=SEED):
    rng=np.random.default_rng(seed); chunks=[]
    for p in paths:
        a=load_patch(p).reshape(13,-1)
        idx=rng.choice(a.shape[1],min(max_pixels,a.shape[1]),replace=False)
        chunks.append(a[:,idx].T.astype(np.float64))
    return np.concatenate(chunks,axis=0)

Xfit=sample_band_matrix(train_fit,CFG["pixels_per_patch_fit"])
band_mean=Xfit.mean(0); band_std=Xfit.std(0)
valid_bands=np.flatnonzero(band_std>1e-8)
assert len(valid_bands)>=CFG["main_k"]
# band vectors are standardized across sampled spatial observations, normalized to unit L2.
Z0=(Xfit[:,valid_bands]-band_mean[valid_bands])/(band_std[valid_bands]+1e-12)
Z=(Z0.T/np.sqrt(max(1,len(Z0)))).astype(np.float64)
Z/=np.linalg.norm(Z,axis=1,keepdims=True)+1e-12
corr=np.clip(Z@Z.T,-1,1)
assert np.allclose(np.sum(Z*Z,axis=1),1,atol=1e-5)
print("valid bands:",valid_bands.tolist(),"correlation mean:",float((corr.sum()-len(corr))/(len(corr)*(len(corr)-1))))

# Group-wise step: K-means on unit-normalized band profiles (Euclidean distance squared = 2(1-r)).
k_main=CFG["main_k"]
km=KMeans(n_clusters=k_main,random_state=SEED,n_init=20)
cluster_labels=km.fit_predict(Z)
# Rank scores computed from train-fit pixels only.
var_scores=Xfit[:,valid_bands].var(0)
entropy_scores=[]
for j in range(len(valid_bands)):
    x=Xfit[:,valid_bands[j]]
    hist,_=np.histogram(x,bins=128)
    prob=hist[hist>0]/hist.sum()
    entropy_scores.append(float(-(prob*np.log2(prob)).sum()))
entropy_scores=np.asarray(entropy_scores)
# Combined ranking score: average percentile ranks of variance and entropy, deterministic.
def percentile_rank_desc(v):
    order=np.argsort(-v,kind="stable"); r=np.empty(len(v),float); r[order]=np.linspace(1,0,len(v)); return r
rank_score=0.5*percentile_rank_desc(var_scores)+0.5*percentile_rank_desc(entropy_scores)
# Candidate pool: top-ranked bands from every group; always include cluster representative.
candidate_pos=[]
group_rows=[]
for c in range(k_main):
    members=np.flatnonzero(cluster_labels==c)
    if len(members)==0: continue
    center=km.cluster_centers_[c]
    rep=members[np.argmin(np.linalg.norm(Z[members]-center,axis=1))]
    ranked=members[np.argsort(-rank_score[members],kind="stable")]
    chosen=list(dict.fromkeys([int(rep)]+ranked[:CFG["cluster_candidates_per_group"]].astype(int).tolist()))
    candidate_pos.extend(chosen)
    group_rows.append(dict(cluster=c,n_members=len(members),representative_band=int(valid_bands[rep]),
                           candidate_bands=[int(valid_bands[x]) for x in chosen]))
candidate_pos=sorted(set(candidate_pos))
candidate_bands=valid_bands[candidate_pos]
print("candidate pool:",candidate_bands.tolist(), "| count:",len(candidate_bands))
pd.DataFrame(group_rows).to_json(OUT/"tables"/"cluster_candidate_groups.json",orient="records",indent=2)
pd.DataFrame({"band":valid_bands,"variance":var_scores,"entropy_bits":entropy_scores,
              "combined_rank_score":rank_score,"cluster":cluster_labels}).to_csv(OUT/"tables"/"band_scores_groups.csv",index=False)

# Train-only regression model: predict omitted original bands from selected decoded bands.
def fit_regression(paths):
    mats=[]
    rng=np.random.default_rng(SEED+11)
    for p in paths:
        a=load_patch(p).reshape(13,-1)
        ix=rng.choice(a.shape[1],min(3000,a.shape[1]),replace=False)
        mats.append(a[:,ix].T.astype(np.float64))
    q=np.concatenate(mats)
    return q.mean(0), np.cov(q,rowvar=False)+1e-5*np.eye(13)
fit_mu,fit_cov=fit_regression(train_fit)
def regression_weights(sel):
    sel=list(sel); mis=[i for i in range(13) if i not in sel]
    if not mis: return mis,None
    A=fit_cov[np.ix_(sel,sel)]+1e-4*np.eye(len(sel))
    W=np.linalg.solve(A,fit_cov[np.ix_(sel,mis)]).T
    intercept=fit_mu[mis]-W@fit_mu[sel]
    return mis,(W,intercept)
def reconstruct_from_decoded(dec,sel):
    """Reconstruct the full 13-band cube from decoded selected bands.

    IMPORTANT: ``sel`` is commonly a tuple of band indices. NumPy interprets
    ``array[tuple]`` as multi-axis indexing, not as a list of row indices.
    Convert it explicitly to a 1-D integer array before indexing the cube.
    """
    dec=np.asarray(dec)
    if dec.ndim != 3 or dec.shape[0] != 13:
        raise ValueError(f"Expected decoded cube with shape (13,H,W); got {dec.shape}")
    sel_idx=np.asarray(list(sel),dtype=np.intp).reshape(-1)
    if sel_idx.size == 0:
        raise ValueError("At least one selected band is required for reconstruction")
    if np.any(sel_idx < 0) or np.any(sel_idx >= dec.shape[0]):
        raise IndexError(f"Selected band indices out of range: {sel_idx.tolist()}")
    if np.unique(sel_idx).size != sel_idx.size:
        raise ValueError(f"Selected band indices must be unique: {sel_idx.tolist()}")

    out=np.empty_like(dec,dtype=np.float64)
    out[sel_idx,:,:]=dec[sel_idx,:,:]
    mis,wi=regression_weights(sel_idx.tolist())
    if mis:
        W,b=wi
        selected_pixels=dec[sel_idx,:,:].reshape(len(sel_idx),-1)
        predicted=W@selected_pixels+b[:,None]
        out[np.asarray(mis,dtype=np.intp),:,:]=predicted.reshape(len(mis),*dec.shape[1:])
    return np.clip(out,0,CFG["psnr_max"])

# Indexing regression test: tuple selections must be treated as band indices,
# not as multi-dimensional NumPy coordinates.
_test_cube=np.zeros((13,4,5),dtype=np.float64)
_test_sel=(0,3,7,9,11,12)
_test_rec=reconstruct_from_decoded(_test_cube,_test_sel)
assert _test_rec.shape==(13,4,5) and np.isfinite(_test_rec).all()
assert np.array_equal(_test_rec[list(_test_sel)],_test_cube[list(_test_sel)])
del _test_cube,_test_rec
print("Reconstruction indexing self-test passed.")

# JPEG2000 codec. Each selected band is independently encoded; byte counts are actual bitstream bytes.
def encode_band(x,rate=CFG["codec_rate"]):
    buf=io.BytesIO()
    Image.fromarray(np.clip(x,0,65535).astype(np.uint16)).save(
        buf,format="JPEG2000",quality_mode="rates",quality_layers=[float(rate)],irreversible=True)
    return buf.getvalue()
def decode_band(blob): return np.asarray(Image.open(io.BytesIO(blob))).astype(np.float64)
def encode_decode_patch(a,sel):
    dec=np.zeros_like(a,dtype=np.float64); nbytes=0; enc=0.; dec_t=0.
    for b in sel:
        t=time.perf_counter(); blob=encode_band(a[b]); enc+=time.perf_counter()-t
        nbytes+=len(blob); t=time.perf_counter(); dec[b]=decode_band(blob); dec_t+=time.perf_counter()-t
    # Regression only uses decoded selected channels.
    rec=reconstruct_from_decoded(dec,sel)
    return rec,nbytes,enc,dec_t

# Rate-aware search fitness: normalised full-spectrum distortion + selected-band codec rate
# + redundancy + a group-coverage penalty. All coefficients are fixed before evaluation.
train_search_sample=train_search[:min(len(train_search),24)]
def make_fitness(paths):
    samples=[]
    for p in paths:
        a=load_patch(p)
        # downsample spatially only for optimisation speed; final metrics use full patches.
        step=max(1,int(max(a.shape[1:])/128))
        samples.append(a[:,::step,::step])
    vari=np.maximum(np.diag(fit_cov),1e-6)
    cache={}
    def fitness(sel):
        sel=tuple(sorted(set(map(int,sel))))
        if len(sel)!=k_main: return 1e9
        if sel in cache:return cache[sel]
        d=[]; byte_frac=[]
        for a in samples:
            rec,nb,_,_=encode_decode_patch(a,sel)
            d.append(np.mean(((a-rec)**2).mean(axis=(1,2))/vari))
            # rate term is bytes vs estimated raw selected-band payload.
            byte_frac.append(nb/(a.shape[1]*a.shape[2]*13*2))
        pos=[int(np.where(valid_bands==b)[0][0]) for b in sel if b in valid_bands]
        red=float(np.mean(np.abs(corr[np.ix_(pos,pos)][np.triu_indices(len(pos),1)]))) if len(pos)>1 else 0.
        groups=set(cluster_labels[pos])
        coverage_penalty=(k_main-len(groups))/k_main
        J=float(np.mean(d)+CFG["mu"]*np.mean(byte_frac)+CFG["nu"]*red+CFG["gamma"]*coverage_penalty)
        cache[sel]=J
        return J
    return fitness
fitness=make_fitness(train_search_sample)

# Simple baselines
def topk(scores,k): return tuple(sorted(valid_bands[np.argsort(-scores,kind="stable")[:k]].tolist()))
def group_representatives(k):
    kmk=KMeans(n_clusters=k,random_state=SEED,n_init=20).fit(Z)
    reps=[]
    for c in range(k):
        ids=np.flatnonzero(kmk.labels_==c); rep=ids[np.argmin(np.linalg.norm(Z[ids]-kmk.cluster_centers_[c],axis=1))]
        reps.append(int(valid_bands[rep]))
    return tuple(sorted(reps))


In [ ]:
# ---------------------------- constrained GA and binary PSO ----------------------------
# Search is restricted to the ranked candidate pool from the group-wise stage.
pool=np.asarray(candidate_bands,dtype=int)
if len(pool)<k_main: raise RuntimeError("Candidate pool smaller than k; increase cluster_candidates_per_group.")
def repair(bits):
    chosen=np.flatnonzero(bits)
    if len(chosen)>k_main:
        chosen=chosen[np.argsort(-rank_score[[np.where(valid_bands==pool[i])[0][0] for i in chosen]])[:k_main]]
    elif len(chosen)<k_main:
        rest=[i for i in np.argsort(-np.array([rank_score[np.where(valid_bands==b)[0][0]] for b in pool])) if i not in chosen]
        chosen=np.r_[chosen,rest[:k_main-len(chosen)]]
    return tuple(sorted(pool[chosen].tolist()))
def random_individual(rng):
    return tuple(sorted(rng.choice(pool,k_main,replace=False).tolist()))
def run_ga(seed):
    rng=np.random.default_rng(seed); pop=[random_individual(rng) for _ in range(CFG["ga_population"])]
    hist=[]; best=None; bestf=np.inf
    for gen in range(CFG["ga_generations"]):
        vals=np.array([fitness(x) for x in pop]); ix=int(np.argmin(vals))
        if vals[ix]<bestf: best,bestf=pop[ix],float(vals[ix])
        hist.append(bestf); new=[best]
        while len(new)<len(pop):
            a,b=rng.choice(len(pop),2,replace=False)
            pa,pb=pop[a],pop[b]
            child=set()
            for g in set(pa)|set(pb):
                if rng.random()<0.5: child.add(g)
            child=list(child)
            if len(child)>k_main: child=list(rng.choice(child,k_main,replace=False))
            if len(child)<k_main:
                rem=[x for x in pool if x not in child]
                child += list(rng.choice(rem,k_main-len(child),replace=False))
            child=list(child)
            if rng.random()<0.25:
                j=int(rng.integers(k_main)); rem=[x for x in pool if x not in child]
                child[j]=int(rng.choice(rem))
            new.append(tuple(sorted(child)))
        pop=new
    return best,bestf,hist
def run_bpso(seed):
    rng=np.random.default_rng(seed); n=len(pool); pos=rng.random((CFG["pso_particles"],n))<k_main/n
    pos=np.asarray([np.isin(pool,repair(row)) for row in pos],dtype=bool)
    vel=np.zeros((CFG["pso_particles"],n)); pbest=pos.copy()
    def score(row): return fitness(tuple(sorted(pool[np.flatnonzero(row)].tolist()))) if row.sum()==k_main else fitness(repair(row))
    pval=np.array([score(x) for x in pos]); g=int(np.argmin(pval)); gbest=pbest[g].copy(); gval=float(pval[g]); hist=[]
    for it in range(CFG["pso_iterations"]):
        for i in range(len(pos)):
            vel[i]=0.7*vel[i]+1.4*rng.random(n)*(pbest[i].astype(float)-pos[i])+1.4*rng.random(n)*(gbest.astype(float)-pos[i])
            prob=1/(1+np.exp(-np.clip(vel[i],-8,8)))
            pos[i]=rng.random(n)<prob
            fixed=repair(pos[i]); pos[i]=np.isin(pool,fixed)
            val=fitness(fixed)
            if val<pval[i]: pbest[i]=pos[i].copy(); pval[i]=val
            if val<gval: gbest=pos[i].copy(); gval=val
        hist.append(gval)
    return tuple(sorted(pool[np.flatnonzero(gbest)].tolist())),gval,hist

search_rows=[]; convergence=[]
for seed in range(SEED,SEED+CFG["ga_pso_seeds"]):
    for name,fn in [("GA",run_ga),("BPSO",run_bpso)]:
        b,v,h=fn(seed)
        search_rows.append(dict(method=name,seed=seed,fitness=v,bands=list(b)))
        for i,x in enumerate(h): convergence.append(dict(method=name,seed=seed,iteration=i+1,best_fitness=x))
search_df=pd.DataFrame(search_rows); search_df.to_csv(OUT/"tables"/"search_runs.csv",index=False)
pd.DataFrame(convergence).to_csv(OUT/"tables"/"search_convergence.csv",index=False)
display(search_df)
# Choose the best search result using validation patches only, not test data.


In [ ]:
# ---------------------------- evaluation metrics ----------------------------
def metric_psnr(x,y):
    mse=float(np.mean((x-y)**2))
    return float("inf") if mse==0 else 10*np.log10(CFG["psnr_max"]**2/mse)
def metric_ssim(x,y):
    vals=[]
    for b in range(13):
        a=x[b]; z=y[b]
        if min(a.shape)<3: continue
        vals.append(structural_similarity(a,z,data_range=CFG["psnr_max"]))
    return float(np.mean(vals)) if vals else np.nan
def metric_sam(x,y):
    a=x.reshape(13,-1).astype(float); b=y.reshape(13,-1).astype(float)
    cos=(a*b).sum(0)/(np.linalg.norm(a,axis=0)*np.linalg.norm(b,axis=0)+1e-12)
    return float(np.degrees(np.arccos(np.clip(cos,-1,1))).mean())
def metric_redundancy(sel):
    ix=[int(np.where(valid_bands==b)[0][0]) for b in sel if b in valid_bands]
    return float(np.mean(np.abs(corr[np.ix_(ix,ix)][np.triu_indices(len(ix),1)]))) if len(ix)>1 else 0.
def evaluate_method(paths,method,sel):
    rows=[]; model_bytes=0; params=0; peak=0
    tracemalloc.start()
    for p in paths:
        a=load_patch(p); rec,nb,enc,dec=encode_decode_patch(a,sel)
        model_bytes += nb; params=(13-len(sel))*(len(sel)+1)
        rows.append(dict(psnr=metric_psnr(a,rec),ssim=metric_ssim(a,rec),sam_deg=metric_sam(a,rec),
                         mae=float(np.mean(np.abs(a-rec))),
                         mre=float(np.mean(np.abs(a-rec)/(np.abs(a)+1.0))),
                         compressed_bytes=nb,enc_s=enc,dec_s=dec,
                         cr=(a.size*CFG["original_bits"])/(nb*8),
                         bpp=(nb*8)/a.size,
                         bits_per_spatial_pixel=(nb*8)/(a.shape[1]*a.shape[2])))
    _,pk=tracemalloc.get_traced_memory();tracemalloc.stop()
    df=pd.DataFrame(rows)
    out={c:float(df[c].mean()) for c in df.columns}
    out.update(method=method,k=len(sel),selected_bands=list(sel),redundancy=metric_redundancy(sel),
               mean_selected_entropy=float(np.mean([entropy_scores[np.where(valid_bands==b)[0][0]] for b in sel if b in valid_bands])),
               regression_parameters=params,peak_traced_memory_MiB=pk/2**20,
               model_size_bytes=int(model_bytes),n_patches=len(paths))
    return out

# Candidate methods include group, ranking, and the proposed hybrid search.
baseline={}
baseline["KMeans_group"]=group_representatives(k_main)
baseline["Variance_rank"]=topk(var_scores,k_main)
baseline["Entropy_rank"]=topk(entropy_scores,k_main)
baseline["Hybrid_group_rank"]=tuple(sorted(int(valid_bands[x]) for x in
    [np.argmin([np.linalg.norm(Z[j]-km.cluster_centers_[c]) for j in range(len(Z)) if cluster_labels[j]==c])
     for c in range(k_main)])) if False else tuple(sorted(int(valid_bands[np.flatnonzero(cluster_labels==c)[np.argmax(rank_score[np.flatnonzero(cluster_labels==c)])]]) for c in range(k_main)))
for name in ["GA","BPSO"]:
    rr=search_df[search_df.method==name].sort_values("fitness")
    if len(rr): baseline["Hybrid_"+name]=tuple(rr.iloc[0].bands)
print("Selection sets:",baseline)

# Validation selects the search strategy; test set is not touched until the winner is frozen.
val_results=[]
for name,sel in baseline.items():
    m=evaluate_method(val_paths,name,sel); val_results.append(m)
val_df=pd.DataFrame(val_results).sort_values(["psnr","sam_deg"],ascending=[False,True])
val_df.to_csv(OUT/"tables"/"validation_comparison.csv",index=False)
display(val_df[["method","k","psnr","ssim","sam_deg","mae","mre","cr","bpp","redundancy","enc_s","dec_s"]])
winner=str(val_df.iloc[0].method)
winner_sel=baseline[winner]
print("Frozen validation winner:",winner,winner_sel)

# Final held-out test evaluation. Also report every comparator on the same test patches.
final_results=[]
for name,sel in baseline.items():
    final_results.append(evaluate_method(final_eval_paths,name,sel))
final_df=pd.DataFrame(final_results).sort_values(["psnr","sam_deg"],ascending=[False,True])
final_df.to_csv(OUT/"tables"/"final_evaluation_comparison.csv",index=False)
display(final_df[["method","k","selected_bands","psnr","ssim","sam_deg","mae","mre","cr","bpp","redundancy","mean_selected_entropy","enc_s","dec_s","regression_parameters","peak_traced_memory_MiB"]])


In [ ]:
# ---------------------------- exports, figures, report ----------------------------
# Selection lists and config are persisted for exact reproducibility.
with open(OUT/"selections"/"selected_band_sets.json","w") as f:
    json.dump({k:list(v) for k,v in baseline.items()},f,indent=2)
with open(OUT/"reports"/"run_config.json","w") as f:
    json.dump({"CFG":CFG,"REGIONS":REGIONS,"winner":winner,"winner_bands":list(winner_sel),
               "data_root":str(DATA_ROOT),"fit_patches":[str(x) for x in train_fit],
               "search_patches":[str(x) for x in train_search],"validation_patches":[str(x) for x in val_paths],
               "official_test_patches":[str(x) for x in test_paths],
               "final_evaluation_patches":[str(x) for x in final_eval_paths],
               "evaluation_label":evaluation_label},f,indent=2)

fig,ax=plt.subplots(figsize=(9,5))
for _,r in final_df.iterrows():
    ax.scatter(r["bpp"],r["psnr"],s=65,label=r["method"])
ax.set_xlabel("Bits per pixel per band (compressed bits / H·W·B)")
ax.set_ylabel("PSNR (dB)");ax.set_title(f"{evaluation_label}: rate–distortion");ax.legend(fontsize=8);plt.tight_layout()
plt.savefig(OUT/"figures"/"final_evaluation_rate_distortion.png",dpi=160);plt.show()

conv=pd.DataFrame(convergence)
if len(conv):
    fig,ax=plt.subplots(figsize=(9,5))
    for method,g in conv.groupby("method"):
        z=g.groupby("iteration").best_fitness.mean()
        ax.plot(z.index,z.values,label=method)
    ax.set_xlabel("Iteration");ax.set_ylabel("Best-so-far fitness");ax.set_title("Search convergence (mean across seeds)");ax.legend()
    plt.tight_layout();plt.savefig(OUT/"figures"/"search_convergence.png",dpi=160);plt.show()

# Classification metrics require a ground-truth class map for the same pixels. DIPCOMA patch
# files used here are image cubes; no label maps are assumed or fabricated.
classification_note=("OA / AA / Cohen's kappa: N/A — this DIPCOMA patch pipeline did not discover "
                     "aligned pixel-level ground-truth class masks. Do not infer classes from region names.")
report=f"""# DIPCOMA hybrid band-selection and compression report

## Protocol
- Dataset root: `{DATA_ROOT}`
- Train fit/search patches: {len(train_fit)} / {len(train_search)}
- Validation patches: {len(val_paths)}
- Final evaluation patches: {len(final_eval_paths)}
- Official test patches: {len(test_paths)}
- Spectral bands: 13; retained bands: {k_main}
- Candidate pool from group→rank stage: {candidate_bands.tolist()}
- Frozen validation winner: **{winner}**, bands `{list(winner_sel)}`

## Final evaluation results
- Evaluation set: {evaluation_label}
- Official test patches available: {len(test_paths)}
{final_df[['method','k','psnr','ssim','sam_deg','mae','mre','cr','bpp','redundancy','enc_s','dec_s','regression_parameters','peak_traced_memory_MiB']].to_markdown(index=False)}

## Metric conventions
- CR = original 16-bit cube bits / actual JPEG2000 selected-band bitstream bits.
- bpp = compressed bits / (H × W × 13), i.e. per-pixel-per-original-band; bits per spatial pixel is separately available in CSV.
- PSNR uses fixed MAX=65535; SSIM is 2-D per band averaged over 13 bands.
- SAM is mean per-pixel spectral angle in degrees; MAE and MRE are full-cube metrics.
- Redundancy is mean absolute Pearson correlation among selected bands.
- OA, AA, κ: {classification_note}
- Encoding/decoding times are measured for selected bands; regression parameter count is reported separately.
- `model_size_bytes` is cumulative encoded bitstream bytes over evaluated patches, not a trained neural model size.
- Peak memory is Python tracemalloc peak for evaluation; it does not include all native-library allocations.

## Interpretation and limitations
The proposed hybrid first groups correlated band profiles, ranks candidates within each group using variance/entropy, then searches a reduced candidate pool with a rate-aware objective. This is an engineering implementation inspired by the group-then-rank hybrid in Vaddi et al. (2024), Algorithm 2, and the rate-aware objective in the supplied Sentinel-2 project note. It is not a reproduction of an exact published algorithm.

Validation-selection regions choose the reported winner; final evaluation uses a region-disjoint set. When the dataset has no official test split, those final results are explicitly a held-out validation-region proxy and must not be presented as official test performance. This is a one-run benchmark, not evidence of statistical significance. For publication, repeat with multiple seeds and report confidence intervals, run a fair identical classifier when valid labels are available, compare more codecs/baselines, and verify dataset acquisition-month metadata against the source files.
"""
(OUT/"reports"/"HYBRID_DIPCOMA_REPORT.md").write_text(report)
print(report)
zip_path=shutil.make_archive("/kaggle/working/hybrid_dipcoma_outputs","zip",OUT)
print("Saved outputs:",OUT,"| ZIP:",zip_path, f"({os.path.getsize(zip_path)/2**20:.1f} MiB)")
